In [1]:
!python --version

Python 3.11.14


## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [2]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### 👉 a TCGA cohort builder + data retrieval engine

Which means you can easily extend to:


> pan-cancer analyses  
> patient-specific pipelines (your perturb_agent 🔥)  
> automated workflows (Nextflow-ready)  

#### 🚀 Next steps

> Production pipeline  
  - CLI tool
  - cached queries
  - reproducible outputs

> Expression matrix builder  
  - auto-download
  - merge TSVs
  - DESeq2-ready matrix

> Patient-level analysis
  - per-case DEG
  - pathway perturbation scoring

### Defaults

In [3]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)[:10]


['subtype_for_TCGA-KIRC.tsv',
 'degs.txt',
 'cases_for_TCGA-LUSC.tsv',
 'subtype_for_TCGA-HNSC.tsv',
 'cases_for_TCGA-PCPG.tsv',
 'Gene_Expression_Quantification_for_TCGA-BRCA_case_0045349c-69d9-4306-a403-c9c1fa836644_sample_type_Primary_Tumor_stage_unknown_fileid_0e0df72c-33c0-4e4f-939c-a4d45a6e1ea3.tsv',
 'cases_for_TCGA-THYM.tsv',
 'cases_for_TCGA-PAAD.tsv',
 'cases_for_TCGA-LAML.tsv',
 'Gene_Expression_Quantification_for_TCGA-BRCA_case_011b9b2d-ebe5-42bf-9662-d922faccc7a1_sample_type_Primary_Tumor_stage_Stage_IA_fileid_89409cf3-e710-47fd-af3c-6f8f0dec7c90.tsv']

### Get all programs

In [4]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/tcga/programs.txt'


In [5]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [6]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [7]:
force=False
verbose=True

program = 'TCGA'

df_psi = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

print(len(df_psi))
df_psi.head(12)


Table opened ((33, 5)) at '/home/flavio/uv/perturb_agent/data/tcga/primary_site_program_TCGA.tsv'
33


,pid,primary_site,project_id,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum,...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neopl...",Bladder Urothelial Carcinoma
3,TCGA-LGG,Brain,TCGA-LGG,Gliomas,Brain Lower Grade Glioma
4,TCGA-GBM,Brain,TCGA-GBM,"Not Reported, Gliomas",Glioblastoma Multiforme
5,TCGA-BRCA,Breast,TCGA-BRCA,"Adnexal and Skin Appendage Neoplasms, Adenomas...",Breast Invasive Carcinoma
6,TCGA-LUAD,Bronchus and lung,TCGA-LUAD,"Ductal and Lobular Neoplasms, Cystic, Mucinous...",Lung Adenocarcinoma
7,TCGA-LUSC,Bronchus and lung,TCGA-LUSC,"Squamous Cell Neoplasms, Adenomas and Adenocar...",Lung Squamous Cell Carcinoma
8,TCGA-MESO,"Bronchus and lung, Heart, mediastinum, and pleura",TCGA-MESO,Mesothelial Neoplasms,Mesothelioma
9,TCGA-CESC,"Cervix uteri, Ovary",TCGA-CESC,"Squamous Cell Neoplasms, Cystic, Mucinous and ...",Cervical Squamous Cell Carcinoma and Endocervi...


In [8]:
df_psi.tail(3)

,pid,primary_site,project_id,disease_type,name
30,TCGA-THCA,Thyroid gland,TCGA-THCA,"Squamous Cell Neoplasms, Epithelial Neoplasms,...",Thyroid Carcinoma
31,TCGA-UCS,"Uterus, NOS, Corpus uteri",TCGA-UCS,"Basal Cell Neoplasms, Complex Mixed and Stroma...",Uterine Carcinosarcoma
32,TCGA-UCEC,"Uterus, NOS, Corpus uteri",TCGA-UCEC,"Not Reported, Cystic, Mucinous and Serous Neop...",Uterine Corpus Endometrial Carcinoma


### Subtypes explanation

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

In [9]:
ipsi = 6
pid = df_psi.iloc[ipsi].pid
primary_site = df_psi.iloc[ipsi].primary_site

print(ipsi, pid, primary_site)

6 TCGA-LUAD Bronchus and lung


- Brain tumors do NOT use AJCC staging (3 - TCGA-LGG Brain)

In [10]:
force=False
verbose=True

pid = df_psi.iloc[ipsi].pid
primary_site = df_psi.iloc[ipsi].primary_site

print(ipsi, pid, primary_site)

df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)

print("df_cases", len(df_cases), "df_subt", len(df_subt))

6 TCGA-LUAD Bronchus and lung
Table opened ((585, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-LUAD.tsv'
df_cases 585 df_subt 23


### df_subt

In [11]:
print(len(df_subt))
df_subt

23


,pid,subtype_global,tumor_class,subtype_tissue,stage,n
0,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,unknown,136
1,TCGA-LUAD,other,other,other,unknown,136
2,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA,76
3,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IB,71
4,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIIA,40
5,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIB,37
6,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIA,27
7,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IV,14
8,TCGA-LUAD,other,other,other,Stage IB,9
9,TCGA-LUAD,other,other,other,Stage IA,8


### df_cases

In [12]:
df_cases.head(3)

,primary_site,disease_type,case_id,diagnoses,pid,subtype_global,stage_ajcc,primary_diagnosis,tumor_grade,stage_clin,...,disease_type_norm,diagnosis_norm,tumor_class,histology,subtype_tissue,consistency,validity,n,frac,sstage
0,Bronchus and lung,Adenomas and Adenocarcinomas,07b5663f-9a54-4462-b6c1-6fc8116b8714,"[{'primary_diagnosis': 'Adenocarcinoma, NOS', ...",TCGA-LUAD,adenocarcinoma_generic,Stage IA,"Adenocarcinoma, NOS",NaN,NaN,...,adenomas and adenocarcinomas,adenocarcinoma,adenocarcinoma,epithelial,lung_adenocarcinoma,ok,valid,1,0.001709,I
1,Bronchus and lung,Adenomas and Adenocarcinomas,294ff941-aea1-4588-9a0e-e9f5393e2bb6,[{'primary_diagnosis': 'Infiltrating duct carc...,TCGA-LUAD,other,Stage IA,"Infiltrating duct carcinoma, NOS",NaN,NaN,...,adenomas and adenocarcinomas,infiltrating duct carcinoma,other,other,other,ok,ambiguous,1,0.001709,I
2,Bronchus and lung,Adenomas and Adenocarcinomas,ebb33753-9d38-4368-9033-3e55f129d00d,[{'primary_diagnosis': 'Adenocarcinoma with mi...,TCGA-LUAD,adenocarcinoma_generic,unknown,Adenocarcinoma with mixed subtypes,NaN,NaN,...,adenomas and adenocarcinomas,adenocarcinoma with mixed subtypes,adenocarcinoma,epithelial,lung_adenocarcinoma,ok,valid,1,0.001709,missing


In [13]:
cols = ["pid", "case_id", "subtype_global", "tumor_class", "subtype_tissue", "stage"]

df_cases[cols].head(12)

,pid,case_id,subtype_global,tumor_class,subtype_tissue,stage
0,TCGA-LUAD,07b5663f-9a54-4462-b6c1-6fc8116b8714,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
1,TCGA-LUAD,294ff941-aea1-4588-9a0e-e9f5393e2bb6,other,other,other,Stage IA
2,TCGA-LUAD,ebb33753-9d38-4368-9033-3e55f129d00d,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,unknown
3,TCGA-LUAD,781f40c9-c099-4c96-8269-ebe2a449c93d,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIB
4,TCGA-LUAD,5134c56f-8286-4ec8-8348-237cee7dad5e,other,other,other,Stage IIA
5,TCGA-LUAD,369f14c4-2191-4962-a309-3e23ddc4e5fc,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIA
6,TCGA-LUAD,e51f717f-72e8-400e-9f35-fc5dc32fdf71,other,other,other,unknown
7,TCGA-LUAD,bbe88801-34f3-46d2-bbfd-b46c3901ed71,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IB
8,TCGA-LUAD,d2c1e896-6886-4122-bb48-5fbcd3f641f4,other,other,other,unknown
9,TCGA-LUAD,108a71cf-b9db-47cd-aa74-c03ec989b41b,other,other,other,Stage IIA


### Samples

In [14]:
force=False
force=False
verbose=True

isubt=0

row = df_subt.iloc[isubt]

pid = row.pid
subtype_global = row.subtype_global
tumor_class = row.tumor_class
subtype_tissue = row.subtype_tissue
stage = None # row.stage

# s_case = f"{pid} subtype '{subtype_global}' tumor '{tumor_class}' subtype_tissue '{subtype_tissue}' stage '{stage}'"
# print(s_case)
df_samples = gdc.get_samples_for_pid_subtypes(pid=pid, subtype_global=subtype_global,
                                             tumor_class=tumor_class, subtype_tissue=subtype_tissue,
                                             batch_size=200, force=force, verbose=verbose)

print(len(df_samples))



Table opened ((585, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-LUAD.tsv'
>>> TCGA-LUAD subtype 'adenocarcinoma_generic' tumor 'adenocarcinoma' subtype_tissue 'lung_adenocarcinoma'
Table opened ((403, 14)) at '/home/flavio/uv/perturb_agent/data/tcga/samples_for_TCGA-LUAD_subtype_'adenocarcinoma_generic'_tumor_'adenocarcinoma'_subtype_tissue_'lung_adenocarcinoma'.tsv'
403


### Available sample types

In [15]:
df_samples.sample_type.unique()

array(['Solid Tissue Normal', 'Primary Tumor', 'Blood Derived Normal'],
      dtype=object)

In [16]:
df2 = df_samples[~df_samples.sample_type.str.contains('Blood', case=False, na=False)]
print(len(df_samples), len(df2))
df2.head()

403 278


,case_id,submitter_id,sample_id,sample_type,barcode_id,file_id,file_name,data_type,data_format,pid,subtype_global,tumor_class,subtype_tissue,stage
0,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,298662ad-dbff-4344-9b80-d744c258c292,Solid Tissue Normal,TCGA-44-2655-11A,8f5ab8b4-72a8-4365-a54a-f7dac0e384bd,3211c900-d9ba-4b44-9329-8bc476c59266.methylati...,Methylation Beta Value,TXT,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
1,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,b2b7acb4-ece5-4e00-bd05-2de5f985bb8a,deae5c6f-6617-4729-98b2-b89ee95cf663.somatic.S...,Structural Rearrangement,BEDPE,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
2,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,87ff6237-1814-4f3b-abd8-d0e0d6f472be,92b25422-ea11-4b60-bc35-e86c0d73db9f.mirbase21...,Isoform Expression Quantification,TXT,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
3,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,5c628172-65f9-4bd7-b608-c24f6745f5ab,af6115c8-f05b-43ea-9e95-5e9cc78e2e89.somatic.S...,Structural Rearrangement,BEDPE,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
4,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,6773903f-ed74-4bb1-8fe4-583ef441861b,HILLY_p_TCGA_b90_wRedos_SNP_N_GenomeWideSNP_6_...,Simple Germline Variation,TSV,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA


In [17]:
df3 = df_samples[df_samples.sample_type == 'Primary Tumor']
print(len(df_samples), len(df2), len(df3))
df3.head()

403 278 252


,case_id,submitter_id,sample_id,sample_type,barcode_id,file_id,file_name,data_type,data_format,pid,subtype_global,tumor_class,subtype_tissue,stage
1,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,b2b7acb4-ece5-4e00-bd05-2de5f985bb8a,deae5c6f-6617-4729-98b2-b89ee95cf663.somatic.S...,Structural Rearrangement,BEDPE,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
2,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,87ff6237-1814-4f3b-abd8-d0e0d6f472be,92b25422-ea11-4b60-bc35-e86c0d73db9f.mirbase21...,Isoform Expression Quantification,TXT,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
3,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,5c628172-65f9-4bd7-b608-c24f6745f5ab,af6115c8-f05b-43ea-9e95-5e9cc78e2e89.somatic.S...,Structural Rearrangement,BEDPE,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
4,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,6773903f-ed74-4bb1-8fe4-583ef441861b,HILLY_p_TCGA_b90_wRedos_SNP_N_GenomeWideSNP_6_...,Simple Germline Variation,TSV,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
5,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,e256a770-8d9c-4fc7-87ea-4f986948407a,HILLY_p_TCGA_b90_wRedos_SNP_N_GenomeWideSNP_6_...,Copy Number Segment,TXT,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA


### CBioPortal studies

In [18]:
study_list = gdc.cbioportal_studies()

study_list.sort()


lista = [x for x in study_list if 'luad' in x]

print("\n".join(lista))

luad_broad
luad_cas_2020
luad_cptac_2020
luad_cptac_gdc
luad_msk_npjpo_2021
luad_mskcc_2015
luad_mskcc_2020
luad_mskcc_2023_met_organotropism
luad_mskimpact_2021
luad_oncosg_2020
luad_tcga
luad_tcga_gdc
luad_tcga_pan_can_atlas_2018
luad_tcga_pub
luad_tsp


### GDC barcodes ~ submitter

In [19]:
barcodes = list(np.unique(df3.barcode_id))
len(barcodes)

10

In [20]:
# if len(barcodes[0].split('-')[-1]) == 3:
#    barcodes = [x[:-1] for x in barcodes]
barcodes

['TCGA-44-2655-01A',
 'TCGA-62-A471-01A',
 'TCGA-62-A471-01Z',
 'TCGA-67-3773-01A',
 'TCGA-67-3773-01Z',
 'TCGA-86-6851-01A',
 'TCGA-86-6851-01Z',
 'TCGA-91-6848-01A',
 'TCGA-NJ-A7XG-01A',
 'TCGA-NJ-A7XG-01Z']

In [21]:
pid

'TCGA-LUAD'

In [22]:
# mat = pid.lower().split('-')
# study_id = mat[1] + '_' + mat[0]
# print(">>>", study_id)

In [23]:
study_id = pid

df_mut = gdc.get_mutations_from_samples(sample_ids=barcodes, study_id=study_id)
len(df_mut)

>>> luad_tcga len = 10 - ['TCGA-44-2655-01', 'TCGA-62-A471-01', 'TCGA-62-A471-01', 'TCGA-67-3773-01', 'TCGA-67-3773-01']...
>>> luad_tcga_pan_can_atlas_2018 len = 10 - ['TCGA-44-2655-01', 'TCGA-62-A471-01', 'TCGA-62-A471-01', 'TCGA-67-3773-01', 'TCGA-67-3773-01']...
>>> https://www.cbioportal.org/api/molecular-profiles/luad_tcga_pan_can_atlas_2018_mutations/mutations/fetch


1773

In [24]:
df_mut.head(3).T

,0,1,2
sampleId,TCGA-44-2655-01,TCGA-44-2655-01,TCGA-44-2655-01
patientId,TCGA-44-2655,TCGA-44-2655,TCGA-44-2655
studyId,luad_tcga_pan_can_atlas_2018,luad_tcga_pan_can_atlas_2018,luad_tcga_pan_can_atlas_2018
molecularProfileId,luad_tcga_pan_can_atlas_2018_mutations,luad_tcga_pan_can_atlas_2018_mutations,luad_tcga_pan_can_atlas_2018_mutations
entrezGeneId,912,1087,1143
proteinChange,L263I,G70V,V253F
mutationType,Missense_Mutation,Missense_Mutation,Missense_Mutation
mutationStatus,.,.,.
variantType,SNP,SNP,SNP
chr,1,19,15


In [27]:
df_mut.columns

Index(['sampleId', 'patientId', 'studyId', 'molecularProfileId',
       'entrezGeneId', 'proteinChange', 'mutationType', 'mutationStatus',
       'variantType', 'chr', 'startPosition', 'endPosition', 'referenceAllele',
       'uniqueSampleKey', 'uniquePatientKey', 'center', 'validationStatus',
       'tumorAltCount', 'tumorRefCount', 'normalAltCount', 'normalRefCount',
       'ncbiBuild', 'keyword', 'variantAllele', 'refseqMrnaId',
       'proteinPosStart', 'proteinPosEnd'],
      dtype='object')

In [25]:
df_mut.groupby(['patientId', 'proteinChange'])

In [ ]:
def map_patient_mutations_simple(df_mut: pd.DataFrame):
    patient_gene_protein = (
        df_mut.groupby("patientId")
        .apply(lambda g: sorted(set(
            f"{row.keyword}:{row.entrezGeneId}:{row.proteinChange}:{row.variantType}:{row.chr}"
            for row in g.itertuples(index=False)
            if pd.notna(row.entrezGeneId) and pd.notna(row.proteinChange)
        )))
        .to_dict()
    )
    return patient_gene_protein

In [35]:
patient_mut_map = map_patient_mutations_simple(df_mut)
patient_mut_map

/tmp/ipykernel_163631/1271118666.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: sorted(set(


{'TCGA-44-2655': ['10527:N973S:SNP:11',
  '10721:D1543H:SNP:3',
  '10758:A52S:SNP:6',
  '10777:P738H:SNP:3',
  '1087:G70V:SNP:19',
  '10880:M41I:SNP:9',
  '11064:E1521*:SNP:9',
  '11085:Q67H:SNP:1',
  '11211:D435Afs*24:DEL:12',
  '112574:E325*:SNP:5',
  '113146:D2351E:SNP:14',
  '1143:V253F:SNP:15',
  '114805:P526T:SNP:2',
  '115019:D362N:SNP:1',
  '115350:P255L:SNP:1',
  '116154:P229T:SNP:20',
  '126205:G582V:SNP:19',
  '127733:P64L:SNP:1',
  '128371:A48E:SNP:1',
  '137970:L704V:SNP:8',
  '140893:D142N:SNP:20',
  '1436:Q448H:SNP:5',
  '1447:A104G:SNP:4',
  '152007:W109L:SNP:9',
  '1663:A572S:SNP:12',
  '169044:X714_splice:SNP:8',
  '1770:I2216N:SNP:17',
  '1903:S361*:SNP:9',
  '200420:X103_splice:SNP:2',
  '2122:N814K:SNP:3',
  '2131:K730Tfs*15:INS:8',
  '222962:A300S:SNP:7',
  '2267:G273C:SNP:8',
  '22873:H361Y:SNP:13',
  '23049:S874C:SNP:16',
  '2317:G1263V:SNP:3',
  '23229:K120M:SNP:23',
  '23236:K1172N:SNP:20',
  '23251:E748K:SNP:15',
  '23269:D2223Y:SNP:15',
  '23345:A15D:SNP:6',